Shallow vs Deep: See the Difference

Compare parameter counts and capabilities of shallow vs deep networks

In [ ]:
import torch
import torch.nn as nn

# shallow - 1 hidden layer learn simple patterns

shallow = nn.Sequential(
    nn.Linear(784,512),
    nn.ReLU(),
    nn.Linear(512,10)
)

# deep- 5 hidden layers learns hierarchical features

deep = nn.Sequential(
    nn.Linear(784,256),# layer 1 - low level features
    nn.ReLU(),
    nn.Linear(256,128), # layer 2 - combine into patterns
    nn.ReLU(),
    nn.Linear(128, 64), # layer 3 - higher abstractions
    nn.ReLU(),
    nn.Linear(64,32), # layer 4 - complex features
    nn.ReLU(),
    nn.Linear(32,10) # layer 5 - classification
)

# compare parameter counts
shallow_params = sum(p.numel() for p in shallow.parameters())
deep_params = sum(p.numel() for p in deep.parameters())

print(f"Shallow network: {shallow_params:,} parameters")
print(f"Deep network:    {deep_params:,} parameters")
print(f"\nKey insight: The deep network has FEWER parameters")
print(f"but can learn MORE complex patterns because each")
print(f"layer builds on the previous one's abstractions.")
print(f"\nShallow = one big step. Deep = many small steps.")


x = torch.randn(1,784)
print(f"\nInput shape: {x.shape}")
print(f"Shallow output: {shallow(x).shape}")
print(f"Deep output: {deep(x).shape}")
# Both produce 10 class scores, but deep features are richer


Shallow network: 407,050 parameters
Deep network:    244,522 parameters

Key insight: The deep network has FEWER parameters
but can learn MORE complex patterns because each
layer builds on the previous one's abstractions.

Shallow = one big step. Deep = many small steps.

Input shape: torch.Size([1, 784])
Shallow output: torch.Size([1, 10])
Deep output: torch.Size([1, 10])


Transfer Learning: The Practical Approach

Use a pre-trained model instead of training from scratch, the standard in 2026




In [3]:
import torch
from torchvision import models, transforms
from torch import nn

# ============================================================
# LOAD A PRE-TRAINED MODEL
# ============================================================
# ResNet-50: 50 layers deep, trained on 1.2M ImageNet images
# The model already learned hierarchical features:
#   Layers 1-15:  edges, textures, colors
#   Layers 16-35: shapes, patterns, parts
#   Layers 36-50: objects, scenes, concepts


model = models.resnet50(weights = 'IMAGENET1K_V2')

# Adapt for your task - transfer learning
# replace only final classification layer
# keep all the learned features frozen initially

num_of_classes = 5 # product categories

# freeze pre-trianed layers

for param in model.parameters():
  param.requires_grad = False

# replace final layer this one will be trained
model.fc = nn.Linear(model.fc.in_features, num_of_classes)

# count parameters
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters:   {total:,}")
print(f"trainable parameters: {trainable:,}")
print(f"frozen parameters: {total - trainable:,}")
print(f"training {trainable/total:1%} of the model!")
print(f"The other {(total-trainable)/total:1%} are pre-trained features")


# ============================================================
# WHY THIS WORKS
# ============================================================
# ImageNet features are general-purpose:
# - Edge detectors transfer to medical images
# - Texture recognizers transfer to satellite imagery
# - Shape detectors transfer to product photos
#
# Rule of thumb:
# - Small dataset (< 1K images): Freeze all, train only fc
# - Medium dataset (1K-10K): Unfreeze last few layers
# - Large dataset (> 10K): Fine-tune entire model with low lr

Total Parameters:   23,518,277
trainable parameters: 10,245
frozen parameters: 23,508,032
training 0.043562% of the model!
The other 99.956438% are pre-trained features


When Not to Use Deep Learning

XGBoost on tabular data often beats deep learning for small datasets




In [4]:
pip install xgboost

In [5]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# ============================================================
# SCENARIO: Small tabular dataset (500 rows, 20 features)
# This is where traditional ML shines!
# ============================================================

X, y = make_classification(n_samples = 500, n_features = 20, n_informative=5, n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X,y, test_size= 0.2, random_state=42
)

# ============================================================
# APPROACH 1: XGBoost (Traditional ML)
# ============================================================
# pip install xgboost

from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators = 100,
    max_depth = 6,
    learning_rate = 0.1,
    random_state= 42
)
xgb.fit(X_train, y_train)
xgb_acc = accuracy_score(y_test, xgb.predict(X_test))


# ============================================================
# APPROACH 2: Simple Neural Network (Deep Learning)
# ============================================================

import torch
import torch.nn as nn

class SimpleNN(nn.Module):

  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(20,64), nn.ReLU(),
        nn.Linear(64,32), nn.ReLU(),
        nn.Linear(32,2)
    )
  def forward(self, x):
    return self.net(x)

torch.manual_seed(42) # Reproducible comparison
model = SimpleNN()
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-3)
criterion = nn.CrossEntropyLoss()

# quick training loop
X_t = torch.FloatTensor(X_train)
y_t = torch.LongTensor(y_train)

for epoch in range(100):
  loss = criterion(model(X_t),y_t)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

model.eval()
with torch.no_grad():
  nn_preds = model(torch.FloatTensor(X_test)).argmax(dim=1).numpy()

nn_acc = accuracy_score(y_test,nn_preds)


# ============================================================
# COMPARE RESULTS
# ============================================================
print(f"XGBoost accuracy:      {xgb_acc:.1%}")
print(f"Neural Net accuracy:   {nn_acc:.1%}")
print(f"\nFor small tabular data, XGBoost often wins because:")
print(f" - Handles feature interactions natively")
print(f" - Robust to missing values and outliers")
print(f" - No GPU needed, trains in seconds")
print(f" - Built-in feature importance for interpretability")


XGBoost accuracy:      92.0%
Neural Net accuracy:   87.0%

For small tabular data, XGBoost often wins because:
 - Handles feature interactions natively
 - Robust to missing values and outliers
 - No GPU needed, trains in seconds
 - Built-in feature importance for interpretability
